<a href="https://colab.research.google.com/github/SujahathMSM/Pytorch-DeepLearning/blob/main/BuildingLLMsFromScratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Data Sampling with Sliding Windows Approach**

In [3]:
!!pip install tiktoken

['Requirement already satisfied: tiktoken in /usr/local/lib/python3.12/dist-packages (0.12.0)',
 'Requirement already satisfied: regex>=2022.1.18 in /usr/local/lib/python3.12/dist-packages (from tiktoken) (2025.11.3)',
 'Requirement already satisfied: requests>=2.26.0 in /usr/local/lib/python3.12/dist-packages (from tiktoken) (2.32.4)',
 'Requirement already satisfied: charset_normalizer<4,>=2 in /usr/local/lib/python3.12/dist-packages (from requests>=2.26.0->tiktoken) (3.4.7)',
 'Requirement already satisfied: idna<4,>=2.5 in /usr/local/lib/python3.12/dist-packages (from requests>=2.26.0->tiktoken) (3.13)',
 'Requirement already satisfied: urllib3<3,>=1.21.1 in /usr/local/lib/python3.12/dist-packages (from requests>=2.26.0->tiktoken) (2.5.0)',
 'Requirement already satisfied: certifi>=2017.4.17 in /usr/local/lib/python3.12/dist-packages (from requests>=2.26.0->tiktoken) (2026.4.22)']

In [4]:
with open ("/content/the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

In [5]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')

In [6]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [7]:
enc_sample = enc_text[50:]

In [8]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287]


In [9]:
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(context, "----->", desired)

[290] -----> 4920
[290, 4920] -----> 2241
[290, 4920, 2241] -----> 287
[290, 4920, 2241, 287] -----> 257


In [10]:
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(tokenizer.decode(context), "----->", tokenizer.decode([desired]))

 and ----->  established
 and established ----->  himself
 and established himself ----->  in
 and established himself in ----->  a


In [11]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

# You need this class defined!
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.tokenizer = tokenizer
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt)

        # Create sliding windows
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


In [12]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

In [13]:
with open ("/content/the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=4,
    stride=1,
    shuffle=False
)

dataiter = iter(dataloader)

first_batch = next(dataiter)
print("first batch----", first_batch)

second_batch = next(dataiter)
print("second batch----", second_batch)

third_batch = next(dataiter)
print("third batch",third_batch)

first batch---- [tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
second batch---- [tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]
third batch [tensor([[2885, 1464, 1807, 3619]]), tensor([[1464, 1807, 3619,  402]])]


In [14]:
with open ("/content/the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=4,
    stride=4,
    shuffle=False
)

dataiter = iter(dataloader)

first_batch = next(dataiter)
print("first batch----", first_batch)

second_batch = next(dataiter)
print("second batch----", second_batch)

third_batch = next(dataiter)
print("third batch",third_batch)

first batch---- [tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]]), tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])]
second batch---- [tensor([[  287,   262,  6001,   286],
        [  465, 13476,    11,   339],
        [  550,  5710,   465, 12036],
        [   11,  6405,   257,  5527],
        [27075,    11,   290,  4920],
        [ 2241,   287,   257,  4489],
        [   64,   319,   262, 34686],
        [41976,    13,   357, 10915]]), tensor([[  262,  6001,   286,   465],
        [

In [15]:
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=4,
    stride=4,
    shuffle=False
)

dataiter = iter(dataloader)

inputs_1, targets_1 = next(dataiter)
inputs_2, targets_2 = next(dataiter)

print("Inputs\n", inputs_1)
print("Targets\n", targets_1)

Inputs
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


**Positional Embeddings**

In [16]:
vocab_size = 50237
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [17]:
token_embedding_layer

Embedding(50237, 256)

In [18]:
token_embedding_layer.weight

Parameter containing:
tensor([[ 2.5222, -1.4666,  0.5432,  ...,  0.2687, -0.9986, -0.6928],
        [-1.5974, -0.0503,  0.4077,  ..., -0.6286, -2.3541, -1.0843],
        [ 0.0484,  1.1515,  1.0811,  ..., -1.0052, -0.4333,  0.5125],
        ...,
        [-0.2687, -0.3451, -0.6003,  ..., -0.3338, -0.8737, -0.0592],
        [ 0.1640,  0.9958, -1.4641,  ..., -1.5143, -0.4587, -0.4640],
        [ 0.1834, -0.5704, -0.3759,  ...,  1.2373, -0.4829,  1.0101]],
       requires_grad=True)

In [19]:
max_lenght = 4
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=max_lenght,
    stride=4,
    shuffle=False
)

In [20]:
dataloader

In [21]:
dataiter = iter(dataloader)
dataiter

In [22]:
next(dataiter)

[tensor([[   40,   367,  2885,  1464],
         [ 1807,  3619,   402,   271],
         [10899,  2138,   257,  7026],
         [15632,   438,  2016,   257],
         [  922,  5891,  1576,   438],
         [  568,   340,   373,   645],
         [ 1049,  5975,   284,   502],
         [  284,  3285,   326,    11]]),
 tensor([[  367,  2885,  1464,  1807],
         [ 3619,   402,   271, 10899],
         [ 2138,   257,  7026, 15632],
         [  438,  2016,   257,   922],
         [ 5891,  1576,   438,   568],
         [  340,   373,   645,  1049],
         [ 5975,   284,   502,   284],
         [ 3285,   326,    11,   287]])]

In [23]:
next(dataiter)

[tensor([[  287,   262,  6001,   286],
         [  465, 13476,    11,   339],
         [  550,  5710,   465, 12036],
         [   11,  6405,   257,  5527],
         [27075,    11,   290,  4920],
         [ 2241,   287,   257,  4489],
         [   64,   319,   262, 34686],
         [41976,    13,   357, 10915]]),
 tensor([[  262,  6001,   286,   465],
         [13476,    11,   339,   550],
         [ 5710,   465, 12036,    11],
         [ 6405,   257,  5527, 27075],
         [   11,   290,  4920,  2241],
         [  287,   257,  4489,    64],
         [  319,   262, 34686, 41976],
         [   13,   357, 10915,   314]])]

In [24]:
len(dataiter)

160

In [25]:
print(f"Total number of batches in the dataloader: {len(dataloader)}")

Total number of batches in the dataloader: 160


In [26]:
input, targets = next(dataiter)
print(input)
print(targets)

tensor([[  314,  2138,  1807,   340],
        [  561,   423,   587, 10598],
        [  393, 28537,  2014,   198],
        [  198,     1,   464,  6001],
        [  286,   465, 13476,     1],
        [  438,  5562,   373,   644],
        [  262,  1466,  1444,   340],
        [   13,   314,   460,  3285]])
tensor([[ 2138,  1807,   340,   561],
        [  423,   587, 10598,   393],
        [28537,  2014,   198,   198],
        [    1,   464,  6001,   286],
        [  465, 13476,     1,   438],
        [ 5562,   373,   644,   262],
        [ 1466,  1444,   340,    13],
        [  314,   460,  3285,  9074]])


In [27]:
print('--- Getting the first batch again ---')
new_dataiter = iter(dataloader) # Create a new iterator
inputs, targets = next(new_dataiter)

print("Inputs (first batch again)\n", inputs)
print("Targets (first batch again)\n", targets)

print('\n--- Now the new_dataiter is at the second batch ---')
second_batch_from_new_iterator_inputs, second_batch_from_new_iterator_targets = next(new_dataiter)
print("Inputs (second batch from new iterator)\n", second_batch_from_new_iterator_inputs)

--- Getting the first batch again ---
Inputs (first batch again)
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets (first batch again)
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])

--- Now the new_dataiter is at the second batch ---
Inputs (second batch from new iterator)
 tensor([[  287,   262,  6001,   286],
        [  465, 13476,    11,   339],
        [  550,  5710,   465, 12036],
        [   11,  6405,   257,  5527],
        [27075,    11,   290,  4920],
        [ 2

In [28]:
token_embeddings = token_embedding_layer(inputs)
print(f"Shape of inputs (token IDs): {inputs.shape}")
print(f"Shape of token_embeddings (embedding vectors): {token_embeddings.shape}")

Shape of inputs (token IDs): torch.Size([8, 4])
Shape of token_embeddings (embedding vectors): torch.Size([8, 4, 256])


In [29]:
print(f"Shape of inputs (token IDs): {inputs.shape}")
print(f"Shape of token_embeddings (embedding vectors): {token_embeddings.shape}")

Shape of inputs (token IDs): torch.Size([8, 4])
Shape of token_embeddings (embedding vectors): torch.Size([8, 4, 256])


In [30]:
print('First 5 rows and first 10 columns of the token embedding matrix:')
print(token_embedding_layer.weight[:5, :10])

First 5 rows and first 10 columns of the token embedding matrix:
tensor([[ 2.5222e+00, -1.4666e+00,  5.4322e-01, -1.3161e+00,  1.7631e-01,
         -6.6808e-01,  9.7199e-01,  8.4632e-01, -1.6640e+00,  7.9856e-02],
        [-1.5974e+00, -5.0286e-02,  4.0769e-01, -4.3482e-02, -4.3909e-01,
          9.6126e-01, -3.5155e-01,  9.3105e-01,  1.1431e+00,  9.3447e-01],
        [ 4.8354e-02,  1.1515e+00,  1.0811e+00,  1.6280e+00,  5.7890e-01,
         -6.9130e-01, -1.7872e+00,  1.2817e-01,  1.5421e+00,  3.7217e-04],
        [ 6.6184e-01,  6.0233e-01,  4.1978e-01, -1.7012e-01,  2.3045e-01,
         -1.3146e+00, -1.7922e+00, -1.4345e+00,  1.4840e-01, -5.3796e-01],
        [ 4.7691e-01, -2.4128e+00,  3.4026e-01,  6.9363e-01, -7.2976e-01,
         -1.2604e+00,  1.0906e+00, -7.8462e-01, -1.0121e+00,  2.1374e-01]],
       grad_fn=<SliceBackward0>)


In [31]:
print('Shape of token_embeddings:', token_embeddings.shape)
print('\nFirst embedding vector in the batch (first token, first sequence):')
print(token_embeddings[0, 0, :])

Shape of token_embeddings: torch.Size([8, 4, 256])

First embedding vector in the batch (first token, first sequence):
tensor([ 1.6968e+00, -9.1428e-01, -7.1353e-01, -9.1430e-01, -1.1169e+00,
        -6.3854e-02,  9.1693e-01, -1.1220e+00, -1.0652e+00, -1.1671e+00,
         6.3497e-01, -2.0403e-01,  7.9327e-01, -7.8136e-02,  1.0821e+00,
         2.1433e-01, -3.7259e-01,  1.9435e-01, -7.9989e-01, -2.0487e-02,
         2.0868e-01, -1.5464e+00, -2.1297e-01,  4.9755e-02,  3.8655e-01,
         1.0252e+00,  7.8620e-01,  4.4258e-01,  4.3606e-01,  2.3482e-01,
        -5.1985e-01,  8.8850e-04, -1.0908e+00,  5.4441e-01, -1.6996e+00,
         1.1393e+00, -9.5535e-02, -1.6351e+00, -1.3660e+00,  3.8363e-01,
        -3.2739e+00,  1.5067e-01,  3.2091e-01, -5.5303e-01, -6.2642e-01,
        -3.1621e-02, -1.1450e+00,  2.0264e-01,  2.1536e-01,  1.1854e+00,
        -6.0145e-01, -5.7353e-01,  6.4628e-01, -7.4793e-02,  8.7334e-01,
         1.5558e+00,  9.7183e-01, -5.0020e-02,  1.7147e+00,  1.7104e-01,
     

In [32]:
context_lenght = max_lenght
pos_embedding_layer = torch.nn.Embedding(context_lenght, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(max_lenght))
print(f"Shape of pos_embeddings: {pos_embeddings.shape}")


Shape of pos_embeddings: torch.Size([4, 256])


In [33]:
input_embeddings = pos_embeddings + token_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
